# AdaSTaR: Adaptive Self-Taught Reasoner for STEM

This notebook implements **AdaSTaR** (Adaptive STaR) with priority-based problem
selection for efficient iterative self-improvement.

**Training pipeline stage:** 3 of 4 (GSPO (curriculum) → RAFT++ → **AdaSTaR** → DPO)

**Target hardware:** Google Colab A100 40GB / 80GB

**Key features (upgrades over vanilla STaR):**
- **Adaptive problem selection:** MinHeap selects stale/hard problems first
- **Staleness tracking:** Problems not solved recently get higher priority
- **Difficulty-aware curriculum weights:** Hard problems weighted more as accuracy grows
- **Domain balancing:** Oversample minority domains in SFT data
- Fresh LoRA adapter on RAFT++ checkpoint (no merge_and_unload)
- `lora_dropout=0.0` for RL consistency

**Algorithm:**
1. Classify problem difficulty (easy/medium/hard)
2. Initialize staleness tracker for all problems
3. For each iteration:
   a. Select top-K problems by priority (staleness + difficulty)
   b. Generate N completions per selected problem
   c. Verify correctness, keep correct completions
   d. Domain-balanced SFT for 1 epoch
   e. Update staleness tracker
   f. Evaluate and check early stop

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

**References:**
- [STaR (arXiv 2203.14465)](https://arxiv.org/abs/2203.14465) — Self-Taught Reasoner
- [ReST (arXiv 2308.08998)](https://arxiv.org/abs/2308.08998) — Reinforced Self-Training
- [AdaSTaR concept](https://arxiv.org/abs/2203.14465) — Adaptive problem selection for efficiency

In [ ]:
# Disable gradient offloading BEFORE importing unsloth
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"

# Install dependencies — TRL >= 0.27.0 pinned for consistency across pipeline
!pip install -q unsloth "trl>=0.27.0" peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy  # For verification

# Login to HuggingFace
from huggingface_hub import login
login()

In [ ]:
# ============================================================
# Add Drive root to sys.path (training/ is at MyDrive level)
# ============================================================
import sys, os

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py", "training/scripts/sort_curriculum.py"]:
    print(f"  {'OK' if os.path.exists(os.path.join(DRIVE_ROOT, s)) else 'MISSING'} {s}")

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model — Instruct base gives dialogue abilities built-in
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
RAFT_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/raft_qwen3_4b/final_adapter"
RAFT_HF_REPO = "Siesher/mits-qwen3-4b-raft"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/star_qwen3_4b"

# ---- A100 GPU preset ----
A100_VRAM_GB = 40

# AdaSTaR parameters
ITERATIONS = 3                    # Maximum iterations
MIN_IMPROVEMENT = 0.02            # Early stop threshold

if A100_VRAM_GB >= 80:
    COMPLETIONS_PER_PROBLEM = 32  # More completions = better coverage
    SFT_BATCH_SIZE = 4
else:
    COMPLETIONS_PER_PROBLEM = 16  # A100 40GB
    SFT_BATCH_SIZE = 2

# Adaptive selection: what fraction of problems to select per iteration
SELECT_TOP_K_RATIO = 0.7         # Select top 70% by priority (skip easy solved ones)

# Generation parameters
MAX_SEQ = 2048
MAX_COMPLETION = 1024
MAX_PROMPT_LENGTH = 512
TEMPERATURE = 0.9                 # Diverse completions
TOP_P = 0.95

# SFT parameters
SFT_EPOCHS = 1
SFT_GRADIENT_ACCUMULATION = 4
SFT_LEARNING_RATE = 1e-5
SFT_WARMUP_STEPS = 20

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0                # No dropout for RL consistency
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Staleness tracker weights
STALENESS_WEIGHT = 0.6            # How much staleness matters
DIFFICULTY_WEIGHT = 0.4           # How much difficulty matters
DIFFICULTY_SCORES = {"easy": 0.2, "medium": 0.5, "hard": 1.0}

# Curriculum weight boost: as accuracy grows, hard problems matter more
CURRICULUM_BOOST_FACTOR = 1.5     # hard_weight = base * (1 + boost * accuracy)

DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]
SYSTEM_PROMPT = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."

print(f"Hardware: A100 {A100_VRAM_GB}GB")
print(f"Base model: {BASE_MODEL}")
print(f"AdaSTaR: {ITERATIONS} max iterations, {COMPLETIONS_PER_PROBLEM} completions/problem")
print(f"Adaptive selection: top {SELECT_TOP_K_RATIO*100:.0f}% by priority")
print(f"Staleness weight: {STALENESS_WEIGHT}, Difficulty weight: {DIFFICULTY_WEIGHT}")
print(f"SFT: lr={SFT_LEARNING_RATE}, epochs={SFT_EPOCHS}, batch={SFT_BATCH_SIZE}")

In [ ]:
# ============================================================
# Mount Drive, resolve checkpoint, load data, load model
# ============================================================
import json
import os
import sys
import random
import math
import heapq
from collections import Counter, defaultdict

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/star_qwen3_4b"

# Resolve RAFT++ checkpoint: Drive first, then download from HuggingFace
if os.path.exists(RAFT_CHECKPOINT):
    print(f"RAFT++ checkpoint found on Drive: {RAFT_CHECKPOINT}")
elif RAFT_HF_REPO:
    from huggingface_hub import snapshot_download
    RAFT_CHECKPOINT = snapshot_download(RAFT_HF_REPO)
    print(f"Downloaded RAFT++ adapter from HuggingFace to: {RAFT_CHECKPOINT}")
else:
    raise FileNotFoundError(f"RAFT++ checkpoint not found: {RAFT_CHECKPOINT}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Load RL problems: Drive JSONL → HF "rl" → HF "gspo" fallback ----
from datasets import load_dataset

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

if os.path.exists(RL_DATA_PATH):
    print(f"Loading RL data from Drive: {RL_DATA_PATH}")
    problems = []
    with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            problems.append(json.loads(line))
    print(f"Loaded {len(problems)} problems from Drive JSONL")
else:
    try:
        print("Trying HF 'rl' config...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
        problems = [dict(r) for r in hf_ds["train"]]
        if "test" in hf_ds:
            problems += [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'rl' config")
    except Exception:
        print("Falling back to HF 'gspo' config...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
        problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'gspo' (fallback)")

verifiable_problems = [
    p for p in problems
    if p.get("type", "verifiable") == "verifiable"
    and p.get("answer_type", "numeric") != "conceptual"
]

print(f"Loaded {len(verifiable_problems)} verifiable problems")
domain_counts = Counter(p.get("domain", "unknown") for p in verifiable_problems)
for d, c in sorted(domain_counts.items()):
    print(f"  {d}: {c}")

# ---- Difficulty classification ----
try:
    from training.scripts.sort_curriculum import classify_difficulty
    print("Imported classify_difficulty")
except ImportError:
    try:
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.sort_curriculum import classify_difficulty
        print("Imported classify_difficulty (via Drive path)")
    except ImportError:
        def classify_difficulty(example):
            text = example.get("answer", "") + " " + example.get("prompt", "")
            word_count = len(text.split())
            if word_count < 50: return "easy"
            elif word_count > 150: return "hard"
            return "medium"

for p in verifiable_problems:
    if "difficulty" not in p:
        p["difficulty"] = classify_difficulty(p)

diff_dist = Counter(p["difficulty"] for p in verifiable_problems)
print(f"\nDifficulty: easy={diff_dist.get('easy',0)}, medium={diff_dist.get('medium',0)}, hard={diff_dist.get('hard',0)}")

# ---- Load model + RAFT++ adapter (auto-detect PEFT) ----
import torch
from unsloth import FastLanguageModel

_raft_source = RAFT_HF_REPO if RAFT_HF_REPO else RAFT_CHECKPOINT
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=_raft_source,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

if hasattr(model, 'peft_config'):
    _cfg = list(model.peft_config.values())[0]
    effective_r = _cfg.r
    raft_alpha = _cfg.lora_alpha
    print(f"RAFT++ adapter loaded from {_raft_source}: r={effective_r}, alpha={raft_alpha}")
else:
    effective_r = LORA_R
    raft_alpha = LORA_ALPHA
    print(f"WARNING: peft_config not found after loading {_raft_source}")

model.gradient_checkpointing_enable()

if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

In [ ]:
# ============================================================
# AdaSTaR utilities: verification, staleness, curriculum, selection
# ============================================================
import re

# ---- Verification functions ----
_verify_imported = False
try:
    from training.scripts.verify_answers import verify, extract_answer
    _verify_imported = True
    print("Imported verify functions from training.scripts.verify_answers")
except ImportError:
    try:
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.verify_answers import verify, extract_answer
        _verify_imported = True
        print("Imported verify functions (via Drive path)")
    except ImportError:
        import sympy
        def extract_answer(text):
            if "</think>" in text: text = text.split("</think>")[-1].strip()
            boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
            return boxed[-1].strip() if boxed else text.strip()
        print("Using fallback verification")


def verify_completion(completion, problem):
    """Verify a completion against ground truth."""
    answer = extract_answer(completion)
    truth = problem.get("ground_truth", problem.get("answer", ""))
    domain = problem.get("domain", "math")
    if _verify_imported:
        result = verify(answer=answer, truth=truth, domain=domain,
                       question_type=problem.get("type", "calc"),
                       test_cases=problem.get("test_cases"))
        return result.correct
    try:
        import sympy
        pred = sympy.sympify(answer)
        gold = sympy.sympify(truth)
        return sympy.simplify(pred - gold) == 0
    except Exception:
        return answer.strip().lower() == str(truth).strip().lower()


# ---- Staleness Tracker ----
class StalenessTracker:
    """Tracks per-problem staleness for adaptive selection.
    Problems not solved recently get higher priority.
    """
    def __init__(self, problems, difficulty_scores, staleness_weight, difficulty_weight):
        self.n_problems = len(problems)
        self.difficulties = [p.get("difficulty", "medium") for p in problems]
        self.difficulty_scores = difficulty_scores
        self.staleness_weight = staleness_weight
        self.difficulty_weight = difficulty_weight
        # last_correct[i] = iteration when problem i was last solved (0 = never)
        self.last_correct = {i: 0 for i in range(self.n_problems)}

    def priority(self, idx, current_iteration):
        """Compute priority score: higher = should be selected first."""
        staleness = current_iteration - self.last_correct[idx]
        diff_score = self.difficulty_scores.get(self.difficulties[idx], 0.5)
        return (self.staleness_weight * staleness +
                self.difficulty_weight * diff_score)

    def update(self, idx, iteration, solved):
        """Update tracker after attempting a problem."""
        if solved:
            self.last_correct[idx] = iteration

    def get_stats(self, current_iteration):
        """Return summary stats for logging."""
        staleness_vals = [current_iteration - self.last_correct[i]
                         for i in range(self.n_problems)]
        return {
            "mean_staleness": sum(staleness_vals) / len(staleness_vals) if staleness_vals else 0,
            "max_staleness": max(staleness_vals) if staleness_vals else 0,
            "never_solved": sum(1 for v in staleness_vals if v == current_iteration),
        }


def curriculum_weight(difficulty, overall_accuracy):
    """Difficulty-aware curriculum weight (GRPO-LEAD style).
    As model improves, hard problems get higher weight.
    """
    base = DIFFICULTY_SCORES.get(difficulty, 0.5)
    if difficulty == "hard":
        return base * (1.0 + CURRICULUM_BOOST_FACTOR * overall_accuracy)
    return base


def adaptive_select_problems(tracker, iteration, top_k_ratio):
    """Select top-K problems by priority (staleness + difficulty).
    Returns (selected_problems, selected_indices).
    """
    n_select = max(1, int(len(verifiable_problems) * top_k_ratio))
    priorities = [(tracker.priority(i, iteration), i) for i in range(len(verifiable_problems))]
    # Use nlargest for efficiency
    top_k = heapq.nlargest(n_select, priorities, key=lambda x: x[0])
    selected_idx = [idx for _, idx in top_k]
    selected = [verifiable_problems[idx] for idx in selected_idx]
    return selected, selected_idx


print(f"AdaSTaR utilities ready")
print(f"  Staleness weight: {STALENESS_WEIGHT}, Difficulty weight: {DIFFICULTY_WEIGHT}")
print(f"  Curriculum boost: {CURRICULUM_BOOST_FACTOR}x for hard at full accuracy")

In [ ]:
# ============================================================
# AdaSTaR Main Loop (with Colab disconnect recovery)
# ============================================================
from trl import SFTConfig, SFTTrainer
from datasets import Dataset


def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect."""
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    path = os.path.join(output_dir, latest)
    print(f"  Found checkpoint: {path}")
    return path


def ada_star_iteration(model, tokenizer, selected_problems, selected_indices,
                       tracker, iteration, overall_accuracy):
    """Run one AdaSTaR iteration on adaptively selected problems."""
    FastLanguageModel.for_inference(model)

    correct_examples = []
    domain_stats = defaultdict(lambda: {"total": 0, "correct": 0, "gen": 0, "gen_correct": 0})

    print(f"  Generating {COMPLETIONS_PER_PROBLEM} completions for {len(selected_problems)} selected problems...")

    for i, (problem, global_idx) in enumerate(zip(selected_problems, selected_indices)):
        domain = problem.get("domain", "math")
        difficulty = problem.get("difficulty", "medium")
        domain_stats[domain]["total"] += 1

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=MAX_PROMPT_LENGTH
        ).to(model.device)

        # Generate completions in batches of 4
        all_completions = []
        generated = 0
        while generated < COMPLETIONS_PER_PROBLEM:
            n_gen = min(4, COMPLETIONS_PER_PROBLEM - generated)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_COMPLETION,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    do_sample=True,
                    num_return_sequences=n_gen,
                )
            prompt_len = inputs["input_ids"].shape[1]
            for j in range(outputs.shape[0]):
                comp = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
                all_completions.append(comp)
            generated += n_gen

        # Verify each completion
        problem_has_correct = False
        for comp in all_completions:
            domain_stats[domain]["gen"] += 1
            if verify_completion(comp, problem):
                domain_stats[domain]["gen_correct"] += 1
                problem_has_correct = True

                # Weight by curriculum
                weight = curriculum_weight(difficulty, overall_accuracy)

                full_messages = messages + [{"role": "assistant", "content": comp}]
                text = tokenizer.apply_chat_template(
                    full_messages, tokenize=False, add_generation_prompt=False
                )
                correct_examples.append({
                    "text": text,
                    "domain": domain,
                    "difficulty": difficulty,
                    "weight": weight,
                })

        # Update staleness tracker
        tracker.update(global_idx, iteration, problem_has_correct)
        if problem_has_correct:
            domain_stats[domain]["correct"] += 1

        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{len(selected_problems)}: {len(correct_examples)} correct completions")

    FastLanguageModel.for_training(model)

    # Domain accuracy
    domain_accuracy = {}
    for domain in DOMAINS:
        s = domain_stats[domain]
        if s["total"] > 0:
            domain_accuracy[domain] = s["correct"] / s["total"]

    return domain_accuracy, correct_examples, dict(domain_stats)


# ---- Initialize staleness tracker ----
tracker = StalenessTracker(
    verifiable_problems, DIFFICULTY_SCORES,
    STALENESS_WEIGHT, DIFFICULTY_WEIGHT,
)

# ---- Resume support: check for previous progress ----
progress_path = os.path.join(OUTPUT_DIR, "progress.json")
start_iter = 0
iteration_results = []
prev_overall_accuracy = 0.0
total_completions_generated = 0
baseline_acc = 0.0

if os.path.exists(progress_path):
    with open(progress_path) as f:
        saved = json.load(f)
    iteration_results = saved.get("iteration_results", [])
    prev_overall_accuracy = saved.get("last_accuracy", 0)
    baseline_acc = saved.get("baseline_acc", 0)
    total_completions_generated = saved.get("total_completions", 0)
    start_iter = len(iteration_results)
    # Restore staleness tracker state
    saved_staleness = saved.get("staleness_last_correct", {})
    if saved_staleness:
        tracker.last_correct = saved_staleness
    if start_iter > 0:
        print(f"Resuming from iteration {start_iter + 1} (prev accuracy: {prev_overall_accuracy:.3f})")
        # Load adapter from last completed iteration
        last_adapter = os.path.join(OUTPUT_DIR, f"iter_{start_iter}_adapter", "adapter_model.safetensors")
        if os.path.exists(last_adapter):
            from safetensors.torch import load_file as _load_iter
            model.load_state_dict(_load_iter(last_adapter), strict=False)
            print(f"  Loaded adapter from iteration {start_iter}")

# ---- Baseline evaluation (only on fresh start) ----
if start_iter == 0:
    print("Evaluating baseline (RAFT++ checkpoint)...")
    FastLanguageModel.for_inference(model)
    baseline_correct = 0
    baseline_total = min(50, len(verifiable_problems))
    baseline_sample = random.sample(verifiable_problems, baseline_total)
    for p in baseline_sample:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": p["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_COMPLETION, temperature=0.7, do_sample=True)
        comp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        if verify_completion(comp, p):
            baseline_correct += 1
    baseline_acc = baseline_correct / baseline_total
    prev_overall_accuracy = baseline_acc
    print(f"Baseline accuracy: {baseline_acc:.3f} ({baseline_correct}/{baseline_total})")
    FastLanguageModel.for_training(model)

# ---- Main AdaSTaR loop ----
for iteration in range(start_iter + 1, ITERATIONS + 1):
    print(f"\n{'='*60}")
    print(f"AdaSTaR Iteration {iteration}/{ITERATIONS}")
    print(f"{'='*60}")

    # Step 1: Adaptive problem selection
    selected, selected_idx = adaptive_select_problems(
        tracker, iteration, SELECT_TOP_K_RATIO
    )
    tier_counts = Counter(p.get("difficulty", "medium") for p in selected)
    print(f"  Selected {len(selected)} problems (of {len(verifiable_problems)})")
    print(f"  By difficulty: easy={tier_counts.get('easy',0)}, medium={tier_counts.get('medium',0)}, hard={tier_counts.get('hard',0)}")

    staleness_stats = tracker.get_stats(iteration)
    print(f"  Staleness: mean={staleness_stats['mean_staleness']:.1f}, max={staleness_stats['max_staleness']}, never_solved={staleness_stats['never_solved']}")

    # Step 2: Generate, verify, collect correct
    domain_acc, correct_examples, gen_stats = ada_star_iteration(
        model, tokenizer, selected, selected_idx,
        tracker, iteration, prev_overall_accuracy,
    )

    total_gen = sum(s["gen"] for s in gen_stats.values())
    total_gen_correct = sum(s["gen_correct"] for s in gen_stats.values())
    total_completions_generated += total_gen
    print(f"\n  Generation: {total_gen} completions, {total_gen_correct} correct ({100*total_gen_correct/max(total_gen,1):.1f}%)")
    print(f"  Correct training examples: {len(correct_examples)}")

    if not correct_examples:
        print("  No correct completions. Stopping.")
        break

    # Step 3: Domain-balanced SFT
    domain_groups = defaultdict(list)
    for ex in correct_examples:
        domain_groups[ex["domain"]].append(ex)

    max_count = max(len(g) for g in domain_groups.values()) if domain_groups else 0
    balanced = []
    for domain, group in domain_groups.items():
        balanced.extend(group)
        if len(group) < max_count:
            balanced.extend(random.choices(group, k=max_count - len(group)))
    random.shuffle(balanced)

    sft_dataset = Dataset.from_list([{"text": ex["text"]} for ex in balanced])
    print(f"  SFT dataset: {len(sft_dataset)} examples (domain-balanced)")

    iter_output = os.path.join(OUTPUT_DIR, f"iteration_{iteration}")
    sft_config = SFTConfig(
        output_dir=iter_output,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRADIENT_ACCUMULATION,
        learning_rate=SFT_LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=SFT_WARMUP_STEPS,
        max_seq_length=MAX_SEQ,
        bf16=True,
        logging_steps=10,
        save_steps=200,
        save_total_limit=1,
        optim="adamw_torch_fused",
        seed=42 + iteration,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )

    # Resume SFT from checkpoint if Colab disconnected during training
    sft_resume = find_latest_checkpoint(iter_output)
    result = trainer.train(resume_from_checkpoint=sft_resume)
    print(f"  SFT loss: {result.training_loss:.4f}")
    del trainer
    torch.cuda.empty_cache()

    # Step 4: Evaluate
    FastLanguageModel.for_inference(model)
    eval_correct = 0
    eval_total = min(50, len(verifiable_problems))
    eval_sample = random.sample(verifiable_problems, eval_total)
    eval_domain_stats = defaultdict(lambda: {"total": 0, "correct": 0})

    for p in eval_sample:
        domain = p.get("domain", "math")
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": p["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_COMPLETION, temperature=0.7, do_sample=True)
        comp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        eval_domain_stats[domain]["total"] += 1
        if verify_completion(comp, p):
            eval_correct += 1
            eval_domain_stats[domain]["correct"] += 1

    FastLanguageModel.for_training(model)

    overall_acc = eval_correct / eval_total
    improvement = overall_acc - prev_overall_accuracy

    print(f"\n  Iteration {iteration} accuracy: {overall_acc:.3f} (delta: {improvement:+.3f})")
    for d in DOMAINS:
        s = eval_domain_stats[d]
        if s["total"] > 0:
            print(f"    {d}: {100*s['correct']/s['total']:.1f}% ({s['correct']}/{s['total']})")

    iteration_results.append({
        "iteration": iteration,
        "accuracy": overall_acc,
        "improvement": improvement,
        "sft_loss": result.training_loss,
        "problems_selected": len(selected),
        "correct_examples": len(correct_examples),
        "total_generated": total_gen,
        "staleness_stats": staleness_stats,
        "tier_distribution": dict(tier_counts),
        "domain_accuracy": dict(eval_domain_stats),
    })

    # Save adapter + progress for Colab disconnect recovery
    iter_adapter_path = os.path.join(OUTPUT_DIR, f"iter_{iteration}_adapter")
    model.save_pretrained(iter_adapter_path)
    tokenizer.save_pretrained(iter_adapter_path)
    with open(progress_path, "w") as f:
        json.dump({
            "iteration_results": iteration_results,
            "last_accuracy": overall_acc,
            "baseline_acc": baseline_acc,
            "total_completions": total_completions_generated,
            "staleness_last_correct": tracker.last_correct,
        }, f, indent=2, default=str)
    print(f"  Progress saved (iteration {iteration})")

    if iteration > 1 and improvement < MIN_IMPROVEMENT:
        print(f"\n  Early stopping: improvement {improvement:.4f} < threshold {MIN_IMPROVEMENT}")
        break

    prev_overall_accuracy = overall_acc

print(f"\nAdaSTaR complete! {len(iteration_results)} iterations")
print(f"Total completions generated: {total_completions_generated}")

In [ ]:
# ============================================================
# Save final adapter + metrics + push to HuggingFace
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final AdaSTaR adapter saved to {final_adapter_path}")

# Save evaluation metrics
eval_metrics = {
    "stage": "ada_star",
    "pipeline_position": "3 of 4",
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "improvement": final_acc - baseline_acc,
    "domain_results": final_domain_results,
    "difficulty_results": final_diff_results,
    "iterations": iteration_results,
    "total_completions_generated": total_completions_generated,
}
eval_path = os.path.join(OUTPUT_DIR, "star_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)
print(f"Eval metrics saved to {eval_path}")

# Save training config
config = {
    "stage": "ada_star",
    "pipeline": "GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "3 of 4",
    "base_model": BASE_MODEL,
    "raft_source": _raft_source,
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "iterations_completed": len(iteration_results),
    "max_iterations": ITERATIONS,
    "completions_per_problem": COMPLETIONS_PER_PROBLEM,
    "select_top_k_ratio": SELECT_TOP_K_RATIO,
    "staleness_weight": STALENESS_WEIGHT,
    "difficulty_weight": DIFFICULTY_WEIGHT,
    "curriculum_boost_factor": CURRICULUM_BOOST_FACTOR,
    "min_improvement": MIN_IMPROVEMENT,
    "sft_lr": SFT_LEARNING_RATE,
    "sft_epochs": SFT_EPOCHS,
    "lora_r": effective_r,
    "lora_alpha": raft_alpha,
    "lora_dropout": LORA_DROPOUT,
    "total_problems": len(verifiable_problems),
    "total_completions_generated": total_completions_generated,
    "difficulty_distribution": dict(diff_dist),
    "references": [
        "STaR arXiv:2203.14465",
        "ReST arXiv:2308.08998",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")

# Push to HuggingFace
PUSH_TO_HUB = True
HF_REPO_ID = "Siesher/mits-qwen3-4b-star"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! AdaSTaR adapter ready for DPO polish (next stage).")

In [ ]:
# ============================================================
# Save final adapter + metrics (backup cell)
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final AdaSTaR adapter saved to {final_adapter_path}")

# Save evaluation metrics
eval_metrics = {
    "stage": "ada_star",
    "pipeline_position": "3 of 4",
    "baseline_accuracy": baseline_acc,
    "final_accuracy": final_acc,
    "improvement": final_acc - baseline_acc,
    "domain_results": final_domain_results,
    "difficulty_results": final_diff_results,
    "iterations": iteration_results,
    "total_completions_generated": total_completions_generated,
}
eval_path = os.path.join(OUTPUT_DIR, "star_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)
print(f"Eval metrics saved to {eval_path}")

# Save training config
config = {
    "stage": "ada_star",
    "pipeline": "GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "3 of 4",
    "base_model": BASE_MODEL,
    "raft_checkpoint": RAFT_CHECKPOINT,
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "iterations_completed": len(iteration_results),
    "max_iterations": ITERATIONS,
    "completions_per_problem": COMPLETIONS_PER_PROBLEM,
    "select_top_k_ratio": SELECT_TOP_K_RATIO,
    "staleness_weight": STALENESS_WEIGHT,
    "difficulty_weight": DIFFICULTY_WEIGHT,
    "curriculum_boost_factor": CURRICULUM_BOOST_FACTOR,
    "min_improvement": MIN_IMPROVEMENT,
    "sft_lr": SFT_LEARNING_RATE,
    "sft_epochs": SFT_EPOCHS,
    "lora_r": effective_r,
    "lora_alpha": raft_alpha,
    "lora_dropout": LORA_DROPOUT,
    "total_problems": len(verifiable_problems),
    "total_completions_generated": total_completions_generated,
    "difficulty_distribution": dict(diff_dist),
    "references": [
        "STaR arXiv:2203.14465",
        "ReST arXiv:2308.08998",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")

# Optional: push to HF
PUSH_TO_HUB = False
HF_REPO_ID = "Siesher/mits-qwen3-4b-star"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! AdaSTaR adapter ready for DPO polish (next stage).")

In [ ]:
# ============================================================
# Post-Training Evaluation on MITS Benchmark
# ============================================================
import sys
sys.path.insert(0, "/content/drive/MyDrive/MITS")

from training.scripts.evaluate_stage import (
    load_eval_dataset, evaluate_with_model, save_report,
    append_summary_csv, print_comparison,
)

STAGE = "star"
EVAL_PATH = "/content/drive/MyDrive/MITS/eval_benchmark.jsonl"
REPORT_DIR = "/content/drive/MyDrive/MITS/evaluation/reports"
SUMMARY_CSV = os.path.join(REPORT_DIR, "summary.csv")

import json, glob
baseline = None
base_reports = sorted(glob.glob(os.path.join(REPORT_DIR, "stage_base_*.json")))
if base_reports:
    with open(base_reports[-1], encoding="utf-8") as f:
        baseline = json.load(f).get("results")

eval_problems = load_eval_dataset(EVAL_PATH)
results = evaluate_with_model(model, tokenizer, eval_problems)

save_report(results, STAGE, "colab", HF_REPO_ID, REPORT_DIR)
append_summary_csv(results, STAGE, SUMMARY_CSV)
print_comparison(results, STAGE, baseline)